# 02 — Candidate retrieval and recall pilot

Run this after **01** has finished. Select **Python (ml-challenge-er)** and run all cells.

This notebook builds a **resumable, disk-backed search index over all training Source 2/3 records**, then retrieves candidates for **5,000 held-out Source 1 businesses**. It uses names, addresses, name character trigrams and joint name/address searches. Retrieval never reads the ground truth; labels are used afterward to measure missed matches.

**Outputs:** a reusable SQLite index, pilot candidate Parquet, per-business recall diagnostics and `pilot_metrics.json`. These are intermediate artifacts, not the final leaderboard TSVs. No XGBoost model is trained here. We need the measured recall before committing to a full training/inference candidate set.

All code lives in the submission `src/` directory. No new Python package or web service is needed. SQLite FTS5 is supplied by the pinned environment. CPU and disk do the work; CUDA is not needed for this stage.

In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "8")
import sys
import json
import sqlite3
import time
import tempfile
import shutil
from pathlib import Path
import polars as pl
from IPython.display import display

# Locate the packaged source whether Jupyter starts in src/, the project, or notebooks/.
ancestors = (Path.cwd(), *Path.cwd().parents)
candidates = [p / relative for p in ancestors for relative in
              (".", "src", "code/business_entity_resolution/src", "submission/code/business_entity_resolution/src")]
SRC = next((p.resolve() for p in candidates if (p / "candidate_retrieval.py").is_file()), None)
assert SRC is not None, "Open this notebook from the submission package or project directory."
sys.path.insert(0, str(SRC))
from candidate_retrieval import cache_parts, build_index, Retriever, sample_anchors, evaluate_pilot, COLUMNS
assert pl.__version__ == "1.44.2", "Use the pinned ml-challenge-er kernel."
assert sqlite3.sqlite_version == "3.53.4", "Use pinned libsqlite=3.53.4; see ../environment.yml."
with sqlite3.connect(":memory:") as check:
    check.execute("CREATE VIRTUAL TABLE check_fts USING fts5(text)")
print("Python:", sys.executable, "| SQLite FTS5:", sqlite3.sqlite_version)


Python: /home/aline/.conda/envs/ml-challenge-er/bin/python | SQLite FTS5: 3.53.4


In [2]:
if os.environ.get("ER_CACHE_ROOT"):
    CACHE_ROOT = Path(os.environ["ER_CACHE_ROOT"]).expanduser().resolve()
else:
    CACHE_ROOT = next((p / "cache/normalized_v1" for p in ancestors
                       if (p / "cache/normalized_v1/train/train_source1/manifest.json").is_file()), None)
assert CACHE_ROOT is not None, "Set ER_CACHE_ROOT to the completed cache from notebook 01."
WORK_ROOT = Path(os.environ.get("ER_CANDIDATE_ROOT", str(CACHE_ROOT.parent / "candidates_v1"))).expanduser().resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_INDEX = WORK_ROOT / "train_search.sqlite"
PILOT_DIR = WORK_ROOT / "pilot"
PILOT_ANCHORS = 5_000
RETRIEVAL = dict(name_k=20, address_k=20, gram_k=10, joint_k=10, max_postings=100_000)
# Union contains at most 60 candidates per anchor; no forced minimum or single best match.
# Changing retrieval settings reuses the index; rerun the pilot before accepting the change.

inventory = []
for split in ("train", "test"):
    for source in (1, 2, 3):
        parts, manifest = cache_parts(CACHE_ROOT, split, source)
        inventory.append({"split": split, "source": source, "rows": manifest["total_rows"], "parts": len(parts)})
label_manifest = json.loads((CACHE_ROOT / "train/train_ground_truth/manifest.json").read_text())
assert label_manifest["complete"], "Ground-truth cache is incomplete."
display(pl.DataFrame(inventory))
print("Working directory:", WORK_ROOT)
print(f"Free disk: {shutil.disk_usage(WORK_ROOT).free / 2**30:.1f} GiB")


split,source,rows,parts
str,i64,i64,i64
"""train""",1,2206821,23
"""train""",2,5034616,51
"""train""",3,5285603,53
"""test""",1,1732544,18
"""test""",2,4887273,49
"""test""",3,5082316,51


Working directory: /home/aline/Projects/ML-Challenge/cache/candidates_v1
Free disk: 66.6 GiB


## Retrieval strategy

An inverted index finds records containing selected name/address tokens without comparing every possible pair. Each query uses rare tokens from the reference record to limit work. Character trigrams (three-letter fragments) give an additional spelling-tolerant search.

The four result sets are **unioned**, so a weak name does not prevent address retrieval. Candidates are restricted to the same country: this is a blocking assumption, not a claim that country labels can never be wrong. The recall audit counts any true matches lost by this assumption or by token selection/top-K truncation.

Default caps: name 20, address 20, trigram 10, joint 10. Posting lists above 100,000 records are excluded to control query cost. These are starting settings, not validated optimal values. Names/addresses with no usable tokens can produce zero candidates.

BM25 scores and retrieval ranks are search evidence, not match probabilities. No postcode or house-number mismatch is used as a hard veto. A generic multilingual model may be worth adding later if these lexical passes miss too many cross-script cases.

In [3]:
# Tiny source fixtures check address rescue, spelling tolerance, no label injection, and resume.
from check_candidates import main as check_retrieval
check_retrieval()


Candidate retrieval, label isolation, recall calculation and index resume checks passed.


## Quick index-size and build-time probe

This temporary index uses 10,000 records from each training target source. It is removed automatically and is never used for recall evaluation. Scaling to 10 million records is not linear: larger posting lists, index merging, and disk traffic can slow the full build. The full build below prints its measured progress and ETA.

In [4]:
with tempfile.TemporaryDirectory(dir=WORK_ROOT, prefix="index-probe-") as temporary:
    probe = Path(temporary)
    probe_rows = 0
    for source in (2, 3):
        parts, _ = cache_parts(CACHE_ROOT, "train", source)
        frame = pl.read_parquet(parts[0], columns=COLUMNS).head(10_000)
        folder = probe / "train" / f"train_source{source}"
        folder.mkdir(parents=True)
        frame.write_parquet(folder / "part.parquet")
        (folder / "manifest.json").write_text(json.dumps({"complete": True, "total_rows": frame.height,
                                                         "parts": {"part.parquet": {"rows": frame.height}}}))
        probe_rows += frame.height
    started = time.perf_counter()
    build_index(probe, "train", probe / "probe.sqlite", progress=False)
    seconds = time.perf_counter() - started
    size = (probe / "probe.sqlite").stat().st_size
    target_rows = sum(r["rows"] for r in inventory if r["split"] == "train" and r["source"] in (2,3))
    estimated_gib = size * target_rows / probe_rows / 2**30
    print(f"Probe: {probe_rows:,} rows in {seconds:.2f}s")
    print(f"Linear build estimate: {seconds * target_rows/probe_rows/60:.1f} min; full-scale indexing can take longer.")
    print(f"Linear index-size estimate: {estimated_gib:.1f} GiB; leave room for WAL/temporary files.")
    assert shutil.disk_usage(WORK_ROOT).free > 2 * estimated_gib * 2**30 + 2**30, "Insufficient estimated working disk. Choose a roomier ER_CANDIDATE_ROOT."


Probe: 20,000 rows in 0.27s
Linear build estimate: 2.4 min; full-scale indexing can take longer.
Linear index-size estimate: 2.1 GiB; leave room for WAL/temporary files.


## Build the full training search index

**Run All executes this cell.** It indexes all 10.32 million training Source 2/3 records. Interrupt if needed; a shard and its checkpoint commit together, so rerunning reuses completed shards. Keep one notebook writer per index path. Do not edit or remove normalized files while indexing.

We build only the training index now. Test-country matching, the full candidate export and classifier training come after checking this pilot. The helper supports a separate test index with the same implementation.

In [5]:
build_summary = build_index(CACHE_ROOT, "train", TRAIN_INDEX)
print(build_summary)


train index: 100,000/10,320,219 | 93,377 rows/s | ETA 1.8 min
train index: 200,000/10,320,219 | 88,255 rows/s | ETA 1.9 min
train index: 300,000/10,320,219 | 84,318 rows/s | ETA 2.0 min
train index: 400,000/10,320,219 | 81,559 rows/s | ETA 2.0 min
train index: 500,000/10,320,219 | 81,909 rows/s | ETA 2.0 min
train index: 600,000/10,320,219 | 81,352 rows/s | ETA 2.0 min
train index: 700,000/10,320,219 | 80,198 rows/s | ETA 2.0 min
train index: 800,000/10,320,219 | 76,445 rows/s | ETA 2.1 min
train index: 900,000/10,320,219 | 73,266 rows/s | ETA 2.1 min
train index: 1,000,000/10,320,219 | 73,133 rows/s | ETA 2.1 min
train index: 1,100,000/10,320,219 | 74,269 rows/s | ETA 2.1 min
train index: 1,200,000/10,320,219 | 74,053 rows/s | ETA 2.1 min
train index: 1,300,000/10,320,219 | 74,222 rows/s | ETA 2.0 min
train index: 1,400,000/10,320,219 | 72,687 rows/s | ETA 2.0 min
train index: 1,500,000/10,320,219 | 71,353 rows/s | ETA 2.1 min
train index: 1,600,000/10,320,219 | 71,694 rows/s | ETA 2.

## Evaluate retrieval on held-out reference businesses

Validation membership is `hash(entity_id, seed=42) % 10 == 0` in the pinned Polars version. The pilot selects 5,000 of those businesses using a second deterministic hash. Preserve this validation split when training the classifier later; do not put these businesses' labeled pairs into training.

The index uses unlabeled target text from the searchable corpus, including validation candidates. It is not trained on validation labels. Candidate selection completes **before** loading labels for evaluation. Singletons remain in the sample.

- **Pair recall:** fraction of all labeled true pairs retrieved.
- **Macro recall:** mean per-business recall, excluding true singletons.
- **Oracle macro F0.5 ceiling:** score if a perfect classifier kept every retrieved true match and rejected every false candidate. This is a ceiling for this candidate set, **not an achieved model score**.
- An empty candidate list is allowed; it is not proof that the business is a singleton.


In [6]:
anchors = sample_anchors(CACHE_ROOT, PILOT_ANCHORS)
retriever = Retriever(TRAIN_INDEX, **RETRIEVAL)
try:
    metrics, details = evaluate_pilot(CACHE_ROOT, retriever, anchors, PILOT_DIR)
finally:
    retriever.close()
print(json.dumps(metrics, indent=2))
print("Saved metrics:", PILOT_DIR / "pilot_metrics.json")
display(details.group_by("country").agg(
    pl.len().alias("anchors"),
    (pl.col("retrieved_matches").sum()/pl.col("true_matches").sum()).alias("pair_recall"),
    pl.col("recall").mean().alias("macro_recall"),
    pl.col("oracle_macro_f05").mean().alias("oracle_f05_ceiling"),
    pl.col("candidates").mean().alias("mean_candidates"),
).sort("country"))
source1_total = next(r["rows"] for r in inventory if r["split"] == "train" and r["source"] == 1)
print(f"Pilot-rate extrapolation for ALL training anchors: {metrics['retrieval_seconds'] / anchors.height * source1_total / 3600:.2f} hours; not scheduled by this notebook.")
print("Candidate counts and this runtime determine the scale of the next training experiment.")


Retrieved 250/5,000 anchors in 195.5s
Retrieved 500/5,000 anchors in 380.0s
Retrieved 750/5,000 anchors in 567.8s
Retrieved 1,000/5,000 anchors in 749.1s
Retrieved 1,250/5,000 anchors in 937.1s
Retrieved 1,500/5,000 anchors in 1124.5s
Retrieved 1,750/5,000 anchors in 1310.0s
Retrieved 2,000/5,000 anchors in 1494.5s
Retrieved 2,250/5,000 anchors in 1683.6s
Retrieved 2,500/5,000 anchors in 1868.0s
Retrieved 2,750/5,000 anchors in 2053.7s
Retrieved 3,000/5,000 anchors in 2242.9s
Retrieved 3,250/5,000 anchors in 2427.6s
Retrieved 3,500/5,000 anchors in 2610.6s
Retrieved 3,750/5,000 anchors in 2798.7s
Retrieved 4,000/5,000 anchors in 2978.5s
Retrieved 4,250/5,000 anchors in 3160.1s
Retrieved 4,500/5,000 anchors in 3340.3s
Retrieved 4,750/5,000 anchors in 3522.7s
Retrieved 5,000/5,000 anchors in 3703.1s
{
  "anchors": 5000,
  "true_pairs": 17417,
  "retrieved_true_pairs": 16700,
  "pair_recall": 0.9588333237641384,
  "macro_recall_non_singletons": 0.9582800862737169,
  "oracle_macro_f05_ceil

country,anchors,pair_recall,macro_recall,oracle_f05_ceiling,mean_candidates
str,u32,f64,f64,f64,f64
"""india""",1954,0.925445,0.922852,0.968752,48.799898
"""us""",3046,0.980498,0.981095,0.994736,47.80302


Pilot-rate extrapolation for ALL training anchors: 454.00 hours; not scheduled by this notebook.
Candidate counts and this runtime determine the scale of the next training experiment.


In [7]:
misses = details.filter(pl.col("retrieved_matches") < pl.col("true_matches"))
print(f"Businesses with at least one missed match: {misses.height:,}/{anchors.height:,}")
display(misses.select("source1_entity_id", "country", "true_matches", "retrieved_matches", "candidates", "missing_true_ids").head(15))
print("Recall settings are experimental. A high oracle ceiling is necessary, but does not prove the classifier will reach it.")
if metrics["pair_recall"] is not None and metrics["pair_recall"] < 0.98:
    print("Candidate recall is below the provisional 98% target. Review missed matches before scaling up.")
else:
    print("Inspect country-specific recall and missing-match examples before moving to model training.")


Businesses with at least one missed match: 550/5,000


source1_entity_id,country,true_matches,retrieved_matches,candidates,missing_true_ids
str,str,i64,i64,i64,list[str]
"""S1-651584324""","""india""",4,3,59,"[""S2-351841986""]"
"""S1-958335738""","""india""",5,3,59,"[""S2-13220684"", ""S2-142339132""]"
"""S1-625160188""","""us""",4,3,53,"[""S3-303667628""]"
"""S1-594787721""","""us""",2,1,48,"[""S2-943271554""]"
"""S1-691987330""","""us""",6,5,42,"[""S2-491140409""]"
…,…,…,…,…,…
"""S1-908628641""","""india""",7,4,54,"[""S2-678984427"", ""S2-87532882"", ""S3-337036587""]"
"""S1-505579021""","""india""",7,3,50,"[""S3-116172547"", ""S3-210669405"", … ""S3-362801624""]"
"""S1-329665807""","""india""",2,1,53,"[""S3-305364502""]"


Recall settings are experimental. A high oracle ceiling is necessary, but does not prove the classifier will reach it.
Candidate recall is below the provisional 98% target. Review missed matches before scaling up.


## What to share / what comes next

Share `pilot_metrics.json` or the printed metrics and country table. Notebook 03 can then construct realistic training pairs and train GPU XGBoost; retrieval settings may first need adjustment based on missed matches.

`pilot_recall.parquet` contains ground-truth diagnostics and must never be consumed as unlabeled inference candidates. `pilot_candidates.parquet` contains only label-blind retrieval output for the pilot, with route bit flags (name=1, address=2, trigrams=4, joint=8), ranks and BM25 scores.

Do not upload any of these intermediate files. The final required `candidate_pairs.tsv` must later represent exactly the **test** pairs actually scored by the final matcher, and `matching_results.tsv` must be its matched subset. France must be retained in that later test stage.

This is a lexical starting point, not a claim of complete multilingual retrieval. It may miss cases where names and addresses both differ heavily after transliteration.